# Training on general (wikilarge) simplification data

In [1]:
#miscellaneous imports
import requests
import os
import datetime
import random
import zipfile
import shutil
import math
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW 
import datasets as hf_datasets 
import transformers as hf_transformers 
from datasets import Dataset, DatasetDict, load_dataset
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, set_seed, get_scheduler
from accelerate import Accelerator, notebook_launcher
from tqdm.notebook import tqdm
import importlib.metadata 
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    set_seed
)
import nltk
import numpy as np

import error: No module named 'triton'


# (Down)load general dataset (larger, not spefici in domain) 
source: 
Xingxing Zhang and Mirella Lapata. 2017. Sentence Simplification with Deep Reinforcement Learning. In Proceedings of the 2017 Conference on Empirical Methods in Natural Language Processing, pages 584–594, Copenhagen, Denmark. Association for Computational Linguistics.

In [2]:

wiki_dataset = load_dataset("bogdancazan/wikilarge-text-simplification")

In [3]:
len(wiki_dataset)

3

In [4]:
#add prefix to the dataset
prefix = "simplify: "
for split in wiki_dataset:
    prefix_column = [prefix] * len(wiki_dataset[split])
    wiki_dataset[split] = wiki_dataset[split].add_column("prefix", prefix_column)
    #rename columns to align with medical training code:
    wiki_dataset[split] = wiki_dataset[split].rename_column("Normal", "input_text")
    wiki_dataset[split] = wiki_dataset[split].rename_column("Simple", "target_text")



In [5]:
wiki_dataset["train"][0] # check the dataset after adding the prefix

{'input_text': 'there is manuscript evidence that austen continued to work on these pieces as late as the period and that her niece and nephew anna and james edward austen made further additions as late as.',
 'target_text': 'there is some proof that austen continued to work on these pieces later in life. her nephew and niece james edward and anna austen may have made further additions to her work in around.',
 'prefix': 'simplify: '}

In [6]:
# encode dataset
#based on https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/T5/Fine_tuning_Dutch_T5_base_on_CNN_Daily_Mail_for_summarization_(on_TPU_using_HuggingFace_Accelerate).ipynb#scrollTo=tiLdcTmkg-_o
from transformers import T5Tokenizer
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-large")

max_input_length = 128
max_target_length = 128

def preprocess_examples(examples):
  input_txt = examples['input_text']
  target_txt = examples['target_text']
  prefix = examples['prefix']
  inputs = [prefix + inp for inp, prefix in zip(input_txt, prefix)]
  model_inputs = tokenizer(inputs, max_length=max_input_length, padding="max_length", truncation=True)
  labels = tokenizer(target_txt, max_length=max_target_length, padding="max_length", truncation=True).input_ids
  labels_with_ignore_index = []
  for labels_example in labels:
    labels_example = [label if label != 0 else -100 for label in labels_example]
    labels_with_ignore_index.append(labels_example)
  
  model_inputs["labels"] = labels_with_ignore_index

  return model_inputs

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [7]:
train_ds = wiki_dataset['train']
val_ds = wiki_dataset['validation']
test_ds = wiki_dataset['test']
encoded_train_ds = train_ds.map(preprocess_examples, batched=True, remove_columns=train_ds.column_names)
encoded_val_ds = val_ds.map(preprocess_examples, batched=True, remove_columns=val_ds.column_names)
encoded_test_ds = test_ds.map(preprocess_examples, batched=True, remove_columns=test_ds.column_names)

In [8]:
encoded_train_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 148843
})

In [9]:
encoded_train_ds['labels'][2]

[116,
 20,
 4401,
 877,
 12,
 8,
 365,
 7276,
 12,
 9635,
 160,
 399,
 7,
 15,
 6399,
 141,
 15,
 7,
 5241,
 399,
 7,
 15,
 6399,
 12,
 3,
 1544,
 8,
 3,
 17043,
 15,
 7662,
 342,
 5,
 227,
 255,
 3,
 342,
 48,
 2728,
 34,
 47,
 5741,
 12,
 453,
 160,
 16,
 8,
 365,
 7276,
 28,
 141,
 15,
 7,
 78,
 255,
 133,
 36,
 5241,
 12,
 20111,
 376,
 5,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100]

In [10]:
encoded_train_ds['input_ids'][3]

[18356,
 10,
 6510,
 900,
 3,
 40,
 52,
 115,
 3,
 52,
 52,
 115,
 19,
 8,
 511,
 167,
 8364,
 1162,
 690,
 16,
 3,
 7,
 15686,
 15,
 7721,
 3,
 40,
 52,
 115,
 227,
 3,
 172,
 2354,
 3,
 52,
 52,
 115,
 11,
 19,
 8,
 167,
 8364,
 1162,
 690,
 13,
 3408,
 2498,
 3,
 40,
 52,
 115,
 8,
 20609,
 4461,
 294,
 13,
 3,
 7,
 15686,
 15,
 7721,
 3,
 52,
 52,
 115,
 5,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

# general setup

In [11]:
model_setup = {
    "model_checkpoint": "google-t5/t5-large",
    "learning_rate": 1e-5, 
    "seed": 42,
}

In [12]:
#set up model and optimizer
import evaluate
set_seed(model_setup["seed"])
model = T5ForConditionalGeneration.from_pretrained(model_setup["model_checkpoint"])
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
optimizer = AdamW(model.parameters(), lr=model_setup["learning_rate"])
#sari_metric = evaluate.load("sari")
nltk.download("punkt", quiet=True)
metric = evaluate.load("rouge")

# Sari metric eval

In [13]:
sari_metric = evaluate.load("sari")
def compute_metrics_sari(eval_preds):
    predictions, labels, inputs = eval_preds.predictions, eval_preds.label_ids, eval_preds.inputs
    """
    Computes SARI
    """
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    inputs = np.where(inputs != -100, inputs, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_inputs = tokenizer.batch_decode(inputs, skip_special_tokens=True)

    all_refs = [[lbl] for lbl in decoded_labels]

    sari_results = sari_metric.compute(
        sources=decoded_inputs,
        predictions=decoded_preds,
        references=all_refs
    )
    return {"sari": sari_results["sari"]}


# Rouge metric eval

In [14]:
# #https://huggingface.co/docs/evaluate/en/transformers_integrations
# import nltk
# nltk.download('punkt_tab')
# def compute_metrics_rouge(eval_preds):
#     preds, labels = eval_preds

#     # Zorg ervoor dat preds en labels geldige token-ID's bevatten
#     preds = np.clip(preds, 0, tokenizer.vocab_size - 1)  # Beperk tot geldige range
#     labels = np.where(labels == -100, tokenizer.pad_token_id, labels)  # Vervang -100 door pad_token_id

#     # Decode de voorspellingen en labels
#     decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

#     # Voeg newlines toe voor rougeLSum
#     decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
#     decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

#     # Bereken de ROUGE-score
#     result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
#     return result

# train for certain hyperparameters, uncomment to run

In [15]:
# training_args = Seq2SeqTrainingArguments(
#     output_dir=hyperparameters["output_dir"],
#     overwrite_output_dir=True,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     num_train_epochs=hyperparameters["num_epochs"],
#     learning_rate=hyperparameters["learning_rate"],
#     lr_scheduler_type="linear",
#     warmup_steps=500,
#     per_device_train_batch_size=16, 
#     per_device_eval_batch_size=16,
#     weight_decay=0.01,
#     predict_with_generate=True,
#     include_inputs_for_metrics=True,
#     gradient_accumulation_steps=2,
#     generation_max_length=128,
#     load_best_model_at_end=True,
#     optim="adamw_torch",
#     fp16=True,
#     label_smoothing_factor=0.1,
#     metric_for_best_model="sari",
#     greater_is_better=True,
#     report_to = "none")

In [16]:
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=encoded_train_ds,
#     eval_dataset=encoded_val_ds,
#     data_collator=data_collator,
#     tokenizer=tokenizer,
#     compute_metrics=compute_metrics_sari
# )

In [17]:
# train_result = trainer.train()
# trainer.save_model(training_args.output_dir)



In [18]:
# import torch
# i=1
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# prompt = f"Simplify text: {multi_cochrane_dataset["train"][i]["input_text"]}"
# inputs = tokenizer(prompt, return_tensors="pt").to(device)
# generated_ids = trainer.model.generate(**inputs, max_length=max_target_length, do_sample=True, top_p=0.95, temperature=0.1)
# summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
# print(f"original: {multi_cochrane_dataset["train"][i]["input_text"]}") 
# print("Simplified:", summary)

# hyperparameter sweep using optuna

In [19]:
import evaluate
set_seed(model_setup["seed"])
model = T5ForConditionalGeneration.from_pretrained(model_setup["model_checkpoint"])
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
optimizer = AdamW(model.parameters(), lr=model_setup["learning_rate"])
sari_metric = evaluate.load("sari")
# nltk.download("punkt", quiet=True)
# metric = evaluate.load("rouge")

In [20]:
hyperparameters = {
    "train_batch_size": 4, # Increased for potential speedup on RTX 4090
    "eval_batch_size": 4, # Increased for potential speedup on RTX 4090
    "seed": 42,
    "mixed_precision": "fp16", 
    "gradient_accumulation_steps": 4, # Adjusted to keep effective batch size 16 (4*4)
}
set_seed(model_setup["seed"])


In [21]:
import optuna

def model_init(trial):
    return T5ForConditionalGeneration.from_pretrained(model_setup["model_checkpoint"])


def hp_space(trial: optuna.Trial): #tried a run with only lr sweep to pinpoint good range
    return {
        "learning_rate": trial.suggest_float("learning_rate", 5e-6, 5e-3, log=True), # Adjusted range for T5-large
        #"num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.2),
        "label_smoothing_factor": trial.suggest_float("label_smoothing_factor", 0.0, 0.3), 
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 1500), 
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine", "constant_with_warmup"]), 
    }

def compute_objective(metrics: dict):
    return metrics.get("eval_sari", 0.0) 

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="D:/ML_Checkpoints/t5_large_general_v3", # Updated output dir for T5-large
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=6,
    per_device_train_batch_size=hyperparameters["train_batch_size"],
    per_device_eval_batch_size=hyperparameters["eval_batch_size"],
    predict_with_generate=True,
    include_for_metrics=["inputs"],
    gradient_accumulation_steps=hyperparameters["gradient_accumulation_steps"],
    generation_max_length=128,
    load_best_model_at_end=True, 
    optim="adamw_torch",
    fp16=False,
    metric_for_best_model="sari",
    greater_is_better=True,      
    report_to="none",              
    seed=hyperparameters["seed"],  
    save_total_limit=1,            
    dataloader_num_workers=4 # Added for potential speedup
)

In [23]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model_init=model_init, 
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_sari
)

In [24]:
N_TRIALS = 25
print(f"Starting hyperparameter search with {N_TRIALS} trials...")

study = optuna.create_study(
    direction="maximize",
    study_name="t5-large-sari-sweep_general_v3", # Updated for T5-large
)

[I 2025-04-15 15:55:09,219] A new study created in memory with name: t5-large-sari-sweep_general_v3


Starting hyperparameter search with 25 trials...


In [ ]:
best_run = trainer.hyperparameter_search(
    direction="maximize",               
    backend="optuna",                   
    hp_space=hp_space,                
    n_trials=N_TRIALS,               
    compute_objective=compute_objective, 
    study_name=study,        
    load_if_exists=True,              

)

[I 2025-04-15 15:55:09,226] A new study created in memory with name: <optuna.study.study.Study object at 0x000001C43C3CED80>
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Sari
1,5.478300,5.532016,25.955825
2,5.765000,5.744675,26.609166


In [ ]:
best_run = trainer.hyperparameter_search(
    direction="maximize",               
    backend="optuna",                   
    hp_space=hp_space,                
    n_trials=N_TRIALS,               
    compute_objective=compute_objective, 
    study_name=study,        
    load_if_exists=True,              

)

[I 2025-04-15 12:51:05,423] A new study created in memory with name: <optuna.study.study.Study object at 0x00000279C3FBA390>


Epoch,Training Loss,Validation Loss,Sari
1,0.000000,nan,40.143316
2,0.000000,nan,40.143316


[W 2025-04-15 15:45:47,260] Trial 0 failed with parameters: {'learning_rate': 3.280586301380341e-05, 'weight_decay': 0.15667339683182457, 'label_smoothing_factor': 0.21116347894282994, 'warmup_steps': 574, 'lr_scheduler_type': 'linear'} because of the following error: SafetensorError('Error while serializing: IoError(Os { code: 112, kind: StorageFull, message: "Onvoldoende schijfruimte beschikbaar." })').
Traceback (most recent call last):
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\integrations\integration_utils.py", line 254, in _objective
    trainer.train(resume_from_checkpoint=checkpoint, trial=trial)
  File "c:\Users\Ruben\GPUcodig\AIenv\Lib\site-packages\transformers\trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Ru

SafetensorError: Error while serializing: IoError(Os { code: 112, kind: StorageFull, message: "Onvoldoende schijfruimte beschikbaar." })

In [ ]:
print("Hyperparameter search finished.")
print(f"Best run found: Run ID {best_run.run_id}") 
print(f"Best SARI score: {best_run.objective}")
print("Best hyperparameters:")
print(best_run.hyperparameters) 
exit()

Hyperparameter search finished.
Best run found: Run ID 3
Best SARI score: 52.94708016500063
Best hyperparameters:
{'learning_rate': 0.0007185024523776204}


In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./optuna_sweep_checkpoints_t5_small_simplification_general_v2/run-3/checkpoint-4652")
hyperparameters = {
    "train_batch_size": 64,
    "eval_batch_size": 64, 
    "seed": 42,
    "mixed_precision": "fp16", 
    "gradient_accumulation_steps": 2,
    'learning_rate': 0.0007185024523776204
}


In [ ]:
print("\nTraining final model with best hyperparameters...")
final_output_dir = "./final_t5_sari_best_model_epoch_15/" 
final_training_args = Seq2SeqTrainingArguments(
    output_dir=final_output_dir,
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=hyperparameters["train_batch_size"],
    per_device_eval_batch_size=hyperparameters["eval_batch_size"],
    predict_with_generate=True,
    include_for_metrics=["inputs"], 
    gradient_accumulation_steps=hyperparameters["gradient_accumulation_steps"],
    generation_max_length=200,
    load_best_model_at_end=True,
    optim="adamw_torch",
    fp16=(hyperparameters["mixed_precision"] == "fp16"),
    metric_for_best_model="sari",
    greater_is_better=True,
    report_to="tensorboard",
    seed=hyperparameters["seed"],
    save_total_limit=2,
    logging_strategy="steps", 
    logging_steps=100,
    num_train_epochs=15,#best_hyperparameters["num_train_epochs"],
    learning_rate=hyperparameters["learning_rate"],
    # lr_scheduler_type=best_hyperparameters["lr_scheduler_type"],
    # warmup_steps=best_hyperparameters["warmup_steps"],
    # weight_decay=best_hyperparameters["weight_decay"],
    # label_smoothing_factor=best_hyperparameters["label_smoothing_factor"],
)

final_trainer = Seq2SeqTrainer(
    model=model_init(None), 
    args=final_training_args,
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_val_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_sari
)

print("Starting final training run...")
train_result = final_trainer.train()
print("Final training complete.")
final_trainer.save_model(final_output_dir)
final_trainer.save_state() 
tokenizer.save_pretrained(final_output_dir)
print(f"Final best model saved to {final_output_dir}")


Training final model with best hyperparameters...


C:\Users\Ruben\AppData\Local\Temp\ipykernel_29644\4090924771.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  final_trainer = Seq2SeqTrainer(


Starting final training run...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Sari
1,1.401600,1.355476,51.691016
2,1.332400,1.324359,52.524581
3,1.279600,1.312462,53.017752
4,1.245500,1.296855,53.329858
5,1.197500,1.292359,52.640593
6,1.166800,1.287182,53.667098
7,1.156800,1.282434,53.013877
8,1.127700,1.278618,53.440163
9,1.106500,1.282740,53.876715
10,1.076800,1.280141,53.869826


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Final training complete.
Final best model saved to ./final_t5_sari_best_model_epoch_25/


In [ ]:
device = torch.cuda.current_device() if torch.cuda.is_available() else "cpu"
model = T5ForConditionalGeneration.from_pretrained(final_output_dir).to(device)

for i in range(10):
    prompt = f"summarize:  {wiki_dataset['train'][i]['input_text']}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    generated_ids = model.generate(**inputs, 
                                   early_stopping= True,
                                   length_penalty= 2.0,
                                   max_length= 200,
                                    min_length= 30,
                                    no_repeat_ngram_size= 3,
                                    num_beams= 4,
                                   temperature=0.7)
    summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    print(f"Original: {wiki_dataset['train'][i]['input_text']}")
    print(f"Simplified: {summary}")

Original: there is manuscript evidence that austen continued to work on these pieces as late as the period and that her niece and nephew anna and james edward austen made further additions as late as.
Simplified: there is some evidence that austen continued to work on these pieces as late as the period and that her niece and nephew anna and james edward austen made further additions.
Original: in a remarkable comparative analysis mandaean scholar s ve s derberg demonstrated that mani s psalms of thomas were closely related to mandaean texts.
Simplified: in a remarkable comparative analysis mandaean scholar s ve s derberg demonstrated that manis psalms of thomas were closely related to mandian texts.
Original: before persephone was released to hermes who had been sent to retrieve her hades tricked her into eating pomegranate seeds lrb six or three according to the telling rrb which forced her to return to the underworld for a period each year.
Simplified: before persephone was released 